In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset, Dataset, WeightedRandomSampler
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from util import filter_data, seed_everything
from util import mask_crop as mask_crop_fn
from validate import val_model, ValLoaderWrapper
from loader import QSM_c1_Dataset as QSM_RAM_Dataset
from networks import QSMDecoder, ResNetWrapper

# ============================================================
# EXPERIMENT CONFIG
# ============================================================
EXP_NAME = "bcond" 
device = 'cuda:0'
LIMIT_SUBS = None 
CACHE_PATH = 'qsm_rep_cache.pt'
LOAD_FROM_CACHE = False  # Build fresh cache first time
seed = 0
seed_everything(0)

# ============================================================
# RUNTIME & CV
# ============================================================
nii_path = '/data2/ali/dbs/qsm/'
seg_path = '/data2/ali/dbs/seg_ps/'
file_dir = '/data2/ali/dbs/dbs_03292024.csv'

cv_features = {'Age', 'Sex', 'Ethnicity', 'Race', 'Disease Duration (year)', 
               ' pre op levadopa equivalent dose (mg)', ' Test medication status', 
               ' OFF (pre-dbs updrs)', ' ON (pre-dbs updrs)'}
all_needed_cols = cv_features | {'CORNELL ID', ' OFF meds ON stim 6mo'}

# ============================================================
# STEP 1: Load ALL subjects with clinical data (for cache building)
# ============================================================
full_clinical_df = filter_data(file_dir, cv_features | {'CORNELL ID'}, True)

# Clinical Normalization on ALL subjects
cols_to_norm = ['Age', 'Disease Duration (year)', ' OFF (pre-dbs updrs)', ' pre op levadopa equivalent dose (mg)']
for col in cols_to_norm:
    full_clinical_df[col] = pd.to_numeric(full_clinical_df[col], errors='coerce')
    col_mean = full_clinical_df[col].mean()
    full_clinical_df[col] = full_clinical_df[col].fillna(col_mean)
    col_std = full_clinical_df[col].std()
    full_clinical_df[col] = (full_clinical_df[col] - col_mean) / (col_std + 1e-8)

# Build clinical_dict for ALL subjects (should give ~108 subjects)
clinical_dict = {str(int(row['CORNELL ID'])): row[list(cv_features)].values.astype(np.float32) 
                 for _, row in full_clinical_df.iterrows()}

print(f"Total subjects with clinical data: {len(clinical_dict)}")

# ============================================================
# STEP 2: Build label_map using ON baseline (to match good cache structure)
# ============================================================
motor_df = filter_data(file_dir, all_needed_cols, True)

# Convert and Clean
for col in [' OFF (pre-dbs updrs)', ' ON (pre-dbs updrs)', ' OFF meds ON stim 6mo']:
    motor_df[col] = pd.to_numeric(motor_df[col], errors='coerce')

motor_df = motor_df.dropna(subset=[' OFF (pre-dbs updrs)', ' OFF meds ON stim 6mo'])

# CACHE BUILDING: Use ON baseline to get 39 non-R / 27 R split
cache_improvement_ratios = (motor_df[' ON (pre-dbs updrs)'] - motor_df[' OFF meds ON stim 6mo']) / motor_df[' ON (pre-dbs updrs)']

cache_label_map = {str(int(row['CORNELL ID'])): (1 if ratio >= 0.30 else 0) 
                   for (_, row), ratio in zip(motor_df.iterrows(), cache_improvement_ratios)}

print(f"Cache label distribution (ON baseline):")
print(f"  Non-Responders: {sum(1 for v in cache_label_map.values() if v == 0)}")
print(f"  Responders: {sum(1 for v in cache_label_map.values() if v == 1)}")
print(f"  Expected: 39 non-R, 27 R (to match good cache)")

# ============================================================
# STEP 3: Build cache with ON-based labels
# ============================================================
full_dataset = QSM_RAM_Dataset(
    nii_path, seg_path, mask_crop_fn, 
    clinical_dict,      # All 108 subjects
    cache_label_map,    # ON-based labels (39 non-R / 27 R)
    limit=LIMIT_SUBS, 
    cache_path=CACHE_PATH, 
    load_cache=LOAD_FROM_CACHE, 
    return_index=True
)

print(f"\nCache built. Verifying structure matches good cache:")
print(f"  Total cached subjects: {len(full_dataset.volumes)}")
print(f"  Total slices: {len(full_dataset.samples)}")

# ============================================================
# STEP 4: NOW use CORRECT OFF-based labels for training
# ============================================================
correct_improvement_ratios = (motor_df[' OFF (pre-dbs updrs)'] - motor_df[' OFF meds ON stim 6mo']) / motor_df[' OFF (pre-dbs updrs)']

# This is the CORRECT label map we actually want to train on
label_map = {str(int(row['CORNELL ID'])): (1 if ratio >= 0.30 else 0) 
             for (_, row), ratio in zip(motor_df.iterrows(), correct_improvement_ratios)}

improvement_ratio_map = {str(int(row['CORNELL ID'])): ratio 
                         for (_, row), ratio in zip(motor_df.iterrows(), correct_improvement_ratios)}

print(f"\nTraining label distribution (OFF baseline - CORRECT):")
print(f"  Non-Responders: {sum(1 for v in label_map.values() if v == 0)}")
print(f"  Responders: {sum(1 for v in label_map.values() if v == 1)}")

actual_clin_dim = next(iter(clinical_dict.values())).shape[0]
full_dataset.clin_dim = actual_clin_dim

# --- SUBJECT SELECTION ---
all_cached_ids = {str(k) for k in full_dataset.volumes.keys()}
label_keys = {str(k) for k in label_map.keys()}
unique_labeled_subs = np.array(sorted(list(all_cached_ids & label_keys)))
unique_sub_labels = np.array([label_map[sid] for sid in unique_labeled_subs])
unlabeled_ids = np.array(list(all_cached_ids - label_keys))

print(f"\nDEBUG: Unique Subjects for CV: {len(unique_labeled_subs)}")
print(f"DEBUG: Total Non-Responders in cohort: {np.sum(unique_sub_labels == 0)}")
print(f"DEBUG: Total Unlabeled subjects: {len(unlabeled_ids)}")

qsm_aug = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05))
])

all_split_best_metrics = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

for split, (t_p_idx, v_p_idx) in enumerate(skf.split(unique_labeled_subs, unique_sub_labels)):
    train_subs, val_subs = unique_labeled_subs[t_p_idx], unique_labeled_subs[v_p_idx]
    
    # Map those subject IDs to ALL corresponding slice indices in the full_dataset
    t_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(train_subs)]
    v_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(val_subs)]
    pt_subs = np.concatenate([unlabeled_ids, train_subs])
    pt_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(pt_subs)]

    print(f"\nSplit {split} | Train Subs: {len(train_subs)}, Val Subs: {len(val_subs)}")
    print(f"  PT Slices: {len(pt_idx)} | FT Slices: {len(t_idx)} | Val Slices: {len(v_idx)}")

    # Sampler for training balance (handles the 5 Non-Responders)
    train_labels = [label_map[str(full_dataset.samples[i]['sub_id'])] for i in t_idx]
    class_counts = np.bincount(train_labels)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    sampler = WeightedRandomSampler([class_weights[l] for l in train_labels], 2*len(t_idx))

    pt_loader = DataLoader(Subset(full_dataset, pt_idx), batch_size=48, shuffle=True)
    t_loader = DataLoader(Subset(full_dataset, t_idx), batch_size=48, sampler=sampler)
    v_loader = DataLoader(Subset(full_dataset, v_idx), batch_size=48, shuffle=False)

    # --- MODEL SETUP ---
    base_resnet = models.resnet18(weights='IMAGENET1K_V1')
    base_resnet.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    model = ResNetWrapper(base_resnet, clinical_dim=actual_clin_dim).to(device)
    decoder = QSMDecoder(feat_dim=512).to(device)

    # --- A. CONDITIONAL PRETRAINING ---
    optimizer_pt = torch.optim.Adam(list(model.base_model.parameters()) + list(decoder.parameters()), lr=1e-4)
    criterion_pt = nn.MSELoss()
    full_dataset.train_mode, full_dataset.transform = True, qsm_aug

    for pt_epoch in range(5):
        model.base_model.train(); decoder.train()
        for imgs, clin, _, _ in pt_loader:
            imgs, clin = imgs.to(device), clin.to(device)
            optimizer_pt.zero_grad()
            _, feats = model(imgs, clin)
            recon = decoder(feats)
            loss = criterion_pt(recon, imgs)
            loss.backward(); optimizer_pt.step()

    # --- B. FINE-TUNING ---
    for param in model.base_model.parameters(): param.requires_grad = False
    optimizer = torch.optim.Adam(model.fusion.parameters(), lr=5e-5, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss().to(device)

    best_f1, best_metrics_this_split, patience = 0, None, 0
    os.makedirs(f"weights/{EXP_NAME}", exist_ok=True)

    for epoch in range(30):
        model.train(); full_dataset.train_mode = True
        for imgs, clin, lbls, _ in t_loader:
            imgs, clin, lbls = imgs.to(device), clin.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits, _ = model(imgs, clin)
            loss_fn(logits, lbls).backward(); optimizer.step()
        
        model.eval(); full_dataset.train_mode, full_dataset.transform = False, None
        wrapped_v_loader = ValLoaderWrapper(v_loader)
        m = val_model(wrapped_v_loader, device, model, loss_fn, v_loader.dataset, threshold=0.5)
        
        current_f1 = 2*(m[2]*m[3])/(m[2]+m[3]) if (m[2]+m[3])>0 else 0

        if current_f1 > best_f1:
            best_f1, best_metrics_this_split, patience = current_f1, m, 0
            torch.save(model.state_dict(), f"weights/{EXP_NAME}/best_f1_split_{split}.pth")
        else:
            patience += 1

        print(f"Split {split} Ep {epoch} | F1: {current_f1:.4f} | AUC: {m[5]:.4f} | Acc: {m[1]:.4f}")
        if patience >= 10: break
    
    if best_metrics_this_split is not None:
        all_split_best_metrics.append(best_metrics_this_split)

# ============================================================
# FINAL SUMMARY
# ============================================================
final_metrics = np.array(all_split_best_metrics)
avg_metrics, std_metrics = np.mean(final_metrics, axis=0), np.std(final_metrics, axis=0)
print("\n" + "="*45 + "\nFINAL CV SUMMARY (SUBJECT-STRATIFIED SLICES)\n" + "="*45)
names = ["Loss", "Accuracy", "Precision", "Sensitivity", "Specificity", "AUC"]
for i, name in enumerate(names):
    print(f"{name:<15} : {avg_metrics[i]:.4f} ± {std_metrics[i]:.4f}")
print("="*45)

Keeping CORNELL ID
Keeping Age
Keeping Sex
Keeping Ethnicity
Keeping Race
Keeping Disease Duration (year)
Keeping  OFF (pre-dbs updrs)
Keeping  ON (pre-dbs updrs)
Keeping  pre op levadopa equivalent dose (mg)
Keeping  Test medication status
Total subjects with clinical data: 69
Keeping CORNELL ID
Keeping Age
Keeping Sex
Keeping Ethnicity
Keeping Race
Keeping Disease Duration (year)
Keeping  OFF (pre-dbs updrs)
Keeping  ON (pre-dbs updrs)
Keeping  pre op levadopa equivalent dose (mg)
Keeping  Test medication status
Keeping  OFF meds ON stim 6mo
Cache label distribution (ON baseline):
  Non-Responders: 40
  Responders: 29
  Expected: 39 non-R, 27 R (to match good cache)


Caching Volumes:   0%|          | 0/111 [00:01<?, ?it/s]


KeyboardInterrupt: 